In [188]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [189]:
DATA_DIR = Path('./data')

In [190]:
mass_towns = gpd.read_file(DATA_DIR / "archive/townssurvey_shp/TOWNSSURVEY_POLYM.shp").to_crs("EPSG:4326")
mass_towns_framingham = mass_towns[mass_towns['TOWN'].str.lower() == 'framingham']
mass_towns_framingham

,TOWN,TOWN_ID,TYPE,COUNTY,FIPS_STCO,FOURCOLOR,POP1960,POP1970,POP1980,POP1990,POP2000,POP2010,POP2020,POPCH10_20,AREA_ACRES,AREA_SQMI,SHAPE_Leng,SHAPE_Area,geometry
177,FRAMINGHAM,100,C,MIDDLESEX,25017,4,44526,64048,65113,64989,66910,68318,72362,4044,16961.824,26.503,35082.75397,6.864206e+07,"POLYGON ((-71.39672 42.34105, -71.39662 42.340..."


In [191]:
addresses = pd.read_excel(DATA_DIR / "archive/MIT Student List 1.27.26.xlsx")

In [192]:
geocoded_addresses = gpd.read_file(DATA_DIR / "archive/MIT Student List 1.27.26 geocoded.csv")
geocoded_addresses = geocoded_addresses.set_geometry(gpd.points_from_xy(
    pd.to_numeric(geocoded_addresses['Geocodio Longitude']),
    pd.to_numeric(geocoded_addresses['Geocodio Latitude'])
), crs="EPSG:4326")

In [193]:
geocoded_in_framingham = gpd.sjoin(mass_towns_framingham, geocoded_addresses, how="inner", predicate='contains')
geocoded_in_framingham.index = geocoded_in_framingham['index_right'].to_list()
geocoded_in_framingham = geocoded_in_framingham.drop(columns=['index_right'])
geocoded_in_framingham.head()

,TOWN,TOWN_ID,TYPE,COUNTY,FIPS_STCO,FOURCOLOR,POP1960,POP1970,POP1980,POP1990,...,Tract median family income as a percentage of the MSA/MD median family income,FFIEC Estimated MSA/MD median family income,Income indicator,CRA poverty criteria?,CRA unemployment criteria?,CRA distressed criteria?,CRA remote rural (low density) criteria?,Previous year CRA distressed criteria?,Previous year CRA underserved criterion?,Meets at least one of current or previous year's CRA distressed/underserved tract criteria?
4698,FRAMINGHAM,100,C,MIDDLESEX,25017,4,44526,64048,65113,64989,...,35.5,146600,Low,No,No,No,No,No,No,No
3778,FRAMINGHAM,100,C,MIDDLESEX,25017,4,44526,64048,65113,64989,...,35.5,146600,Low,No,No,No,No,No,No,No
8551,FRAMINGHAM,100,C,MIDDLESEX,25017,4,44526,64048,65113,64989,...,35.5,146600,Low,No,No,No,No,No,No,No
8110,FRAMINGHAM,100,C,MIDDLESEX,25017,4,44526,64048,65113,64989,...,35.5,146600,Low,No,No,No,No,No,No,No
761,FRAMINGHAM,100,C,MIDDLESEX,25017,4,44526,64048,65113,64989,...,35.5,146600,Low,No,No,No,No,No,No,No


In [194]:
geocoded_in_framingham.index

Index([4698, 3778, 8551, 8110,  761, 3250, 4109, 4647, 6821, 6165,
       ...
       4076, 7141, 7140, 1850, 8191,  106, 8052, 5665, 5466, 5498],
      dtype='int64', length=8716)

In [195]:
merged_addresses = addresses.merge(geocoded_in_framingham, left_index=True, right_index=True, how='left', suffixes=['', '_geocodio'])

In [196]:
disparities = merged_addresses[merged_addresses['Address'] != merged_addresses['Address_geocodio']].copy()
disparities.shape

(144, 742)

In [197]:
address_check_completed = gpd.read_file(DATA_DIR / "archive/address_check_completed.csv")
address_check_completed = address_check_completed.set_geometry(gpd.points_from_xy(
    pd.to_numeric(address_check_completed['Geocodio Longitude']),
    pd.to_numeric(address_check_completed['Geocodio Latitude'])
), crs="EPSG:4326")
address_check_completed.index = disparities.index
address_check_completed.head()

,Address,Geocodio Latitude,Geocodio Longitude,Geocodio Accuracy Score,Geocodio Accuracy Type,Geocodio Address Line 1,Geocodio Address Line 2,Geocodio Address Line 3,Geocodio House Number,Geocodio Street,...,"ACS Families/Own Children Under 18 Years by Family Type and Age/In other families: Female householder, no spouse present: 5 years/Value","ACS Families/Own Children Under 18 Years by Family Type and Age/In other families: Female householder, no spouse present: 5 years/Margin of error","ACS Families/Own Children Under 18 Years by Family Type and Age/In other families: Female householder, no spouse present: 5 years/Percentage","ACS Families/Own Children Under 18 Years by Family Type and Age/In other families: Female householder, no spouse present: 6 to 11 years/Value","ACS Families/Own Children Under 18 Years by Family Type and Age/In other families: Female householder, no spouse present: 6 to 11 years/Margin of error","ACS Families/Own Children Under 18 Years by Family Type and Age/In other families: Female householder, no spouse present: 6 to 11 years/Percentage","ACS Families/Own Children Under 18 Years by Family Type and Age/In other families: Female householder, no spouse present: 12 to 17 years/Value","ACS Families/Own Children Under 18 Years by Family Type and Age/In other families: Female householder, no spouse present: 12 to 17 years/Margin of error","ACS Families/Own Children Under 18 Years by Family Type and Age/In other families: Female householder, no spouse present: 12 to 17 years/Percentage",geometry
5,"353 Old Connecticut Path Framingham, MA 01701,...",42.308843,-71.398347,1,rooftop,353 Old Connecticut Path,,"Framingham, MA 01701",353,Old Connecticut Path,...,12,See unified,,,,See unified,,,,POINT (-71.39835 42.30884)
43,"695 Old Connecticut Path Framingham MA 01701, USA",42.318237,-71.391734,1,rooftop,695 Old Connecticut Path,,"Framingham, MA 01701",695,Old Connecticut Path,...,0,14,0,12,21,1,0,14,0,POINT (-71.39173 42.31824)
252,"69 2nd St #69b, Framingham, MA 01702, USA",42.276886,-71.400199,1,rooftop,69 2nd St,# 69B,"Framingham, MA 01702",69,2nd St,...,0,14,0,209,114,0.303,355,171,0.514,POINT (-71.4002 42.27689)
749,"1 Pine Hill Rd, Framingham, MA 01701, USA",42.271945,-71.419755,0.91,nearest_rooftop_match,2 Pine Pl,,"Framingham, MA 01702",2,Pine Pl,...,0,14,0,47,76,0.287,61,101,0.372,POINT (-71.41976 42.27194)
778,"154 Apricot St, Worcester, MA 01603, USA",42.242796,-71.86246,1,rooftop,154 Apricot St,,"Worcester, MA 01603",154,Apricot St,...,0,14,0,52,49,0.547,38,32,0.4,POINT (-71.86246 42.2428)


In [198]:
# Replace the indices in merged_addresses with those from address_check_completed
merged_addresses.update(address_check_completed)
merged_addresses = merged_addresses.drop(columns=['Address_geocodio'])

# Forward fill the columns related to Framingham town data
merged_addresses[mass_towns_framingham.columns] = merged_addresses[mass_towns_framingham.columns].ffill()

In [ ]:
merged_addresses.to_csv(DATA_DIR / "geocoded_addresses_final.csv", index=False)

In [200]:
set(address_check_completed.columns.to_list()).symmetric_difference(set(geocoded_addresses.columns.to_list()))

set()

# Schools

In [201]:
framingham_schools = addresses['School Code'].unique()
framingham_schools

<StringArray>
['THAYER CAMPUS',           'BAR',           'BRO',           'CAM',
           'DUN',           'FHC',           'FHE',           'FHS',
           'FUL',           'HAR',           'HAS',           'HEM',
           'HNA',           'JUN',           'KNG',           'MCC',
           'PEF',           'PEL',           'PEY',           'POT',
           'PRI',           'SRF',           'STA',           'THE',
           'WAL']
Length: 25, dtype: str

In [204]:
geocoded_schools = pd.read_csv(DATA_DIR / "geocoded_schools.csv", encoding='latin-1')
geocoded_schools = gpd.GeoDataFrame(geocoded_schools, geometry=gpd.points_from_xy(
    pd.to_numeric(geocoded_schools['Geocodio Longitude']),
    pd.to_numeric(geocoded_schools['Geocodio Latitude'])
), crs="EPSG:4326")
geocoded_schools.head()

,School Code,Name,Address,Geocodio Latitude,Geocodio Longitude,Geocodio Accuracy Score,Geocodio Accuracy Type,Geocodio Address Line 1,Geocodio Address Line 2,Geocodio Address Line 3,...,Unified School District Grade High,Elementary School District Name,Elementary School District LEA code,Elementary School District Grade Low,Elementary School District Grade High,Secondary School District Name,Secondary School District LEA code,Secondary School District Grade Low,Secondary School District Grade High,geometry
0,THAYER CAMPUS,Framingham High School - Thayer Campus,"50 Lawrence St, Framingham, MA 01702",42.282753,-71.411640,1.00,rooftop,50 Lawrence St,NaN,"Framingham, MA 01702",...,12.0,See unified,NaN,NaN,NaN,See unified,NaN,NaN,NaN,POINT (-71.41164 42.28275)
1,BAR,Barbieri Elementary School,"100 Dudley Rd, Framingham, MA 01702",42.279627,-71.432341,1.00,rooftop,100 Dudley Rd,NaN,"Framingham, MA 01702",...,12.0,See unified,NaN,NaN,NaN,See unified,NaN,NaN,NaN,POINT (-71.43234 42.27963)
2,BRO,Brophy Elementary School,"575 Pleasant St, Framingham, MA 01701",42.306510,-71.463880,1.00,rooftop,575 Pleasant St,NaN,"Framingham, MA 01701",...,12.0,See unified,NaN,NaN,NaN,See unified,NaN,NaN,NaN,POINT (-71.46388 42.30651)
3,CAM,Cameron Middle School,"215 Elm St, Framingham, MA 01701",42.333209,-71.399944,1.00,rooftop,215 Elm St,NaN,"Framingham, MA 01701",...,12.0,See unified,NaN,NaN,NaN,See unified,NaN,NaN,NaN,POINT (-71.39994 42.33321)
4,DUN,Charlotte A. Dunning Elementary School,"48 Frost St, Framingham, MA 01702",42.321515,-71.430777,0.99,rooftop,48 Frost St,NaN,"Framingham, MA 01701",...,12.0,See unified,NaN,NaN,NaN,See unified,NaN,NaN,NaN,POINT (-71.43078 42.32152)
